# Day 2 Lab — MLPs & Backpropagation

**Learning goals**
1. See derivatives and gradient descent in pictures, and check them in Python.
2. Follow backpropagation step by step on a tiny network and confirm it with PyTorch.
3. Build and train your first MLP in PyTorch (spirals) and see what hidden size and activation do.
4. Train PyTorch MLPs on FashionMNIST and compare width, depth and activation functions.
5. Train an MLP seizure detector on EEG band-power features (CHB-MIT) and see why accuracy is misleading.

**Colab:** *Runtime → Change runtime type → T4 GPU* (only Part C really needs it; everything also runs on a CPU).

Plan: Part 0 (15 min) → Part A (25 min) → Part B (30 min) → Part C (35 min) → Part D (25 min, the rest is homework)
→ ⭐ Part E (optional, for the curious).

### How to work with this notebook
Most cells are complete examples. You do **not** have to write code from scratch.

| symbol | what you do |
|---|---|
| ▶️ **Run** | run the cell and read the comments and the output |
| 🤔 **Predict first** | write down what you expect *before* you run the cell |
| 🔧 **Change it** | change one number or one word, run again, compare |
| ✍️ **Fill in** | write 1-3 lines where it says `# YOUR CODE HERE`; the next cell prints ✅ or ❌ |
| 💬 **Explain** | answer in 1-3 sentences in the text cell |
| ⭐ **optional** | only if you have time (or at home) |
| 🧰 **Helpers** | just run the cell - you do not need to read it |

### Mini glossary
- **tensor** - PyTorch's name for an array of numbers (like a NumPy array), e.g. a batch of images.
- **shape** - the size of a tensor in every direction, e.g. `(256, 784)` = 256 rows (samples) with 784 numbers each.
- **`nn.Linear(a, b)`** - a layer of `b` neurons; each computes a weighted sum of `a` inputs plus a bias.
- **`nn.Sequential(...)`** - puts layers in a row: the output of one layer is the input of the next.
- **activation** - the function after the weighted sum, e.g. ReLU (negative → 0, positive stays).
- **loss** - one number that says how wrong the model is; training makes it small.
- **optimizer** - the part that changes the weights, e.g. `SGD` = gradient descent.
- **batch** - a small group of samples that is processed together in one step.
- **epoch** - one pass through all training samples.
- **logits** - the raw output scores of the last layer, one per class (before the softmax).
- **accuracy** - the fraction of samples whose predicted class is correct.

### Help: common errors
- `NameError: name 'x' is not defined` → you skipped a cell above. Run all cells from the top: *Runtime → Run before*.
- `RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x90 and 784x256)` → the shapes do not fit.
  Print `.shape` of every tensor in the line. The number of inputs of a layer must match the data.
- `expected scalar type Float but found Long` / `Double` → mixed number types. Use `torch.tensor(..., dtype=torch.float32)`
  for inputs and `torch.long` for class labels.
- `Expected all tensors to be on the same device` → the model is on `cuda`, the data on `cpu` (or the other way round).
  Move both with `.to(device)`.
- Something strange after many changes? *Runtime → Restart session* and run again from the top.

🧰 **Setup - just run this cell.** It loads the libraries, chooses the GPU if there is one and defines `check`,
which prints ✅ or ❌ after a fill-in.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__, "| device:", device)


def check(name, value, expected, hint, tol=1e-6):
    """Print ✅ if value is (numerically) equal to expected, otherwise ❌ and a hint."""
    if value is None:
        print(f"❌ not yet - {name} is still None. Hint: {hint}")
        return
    try:
        ok = np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol)
    except (TypeError, ValueError):  # e.g. wrong shape
        ok = False
    if ok:
        print(f"✅ correct: {name} = {np.round(np.asarray(value, dtype=float), 4)}")
    else:
        print(f"❌ not yet - {name} = {value}. Hint: {hint}")

## Part 0 — Warm-up: derivatives with numbers
⏱ about 15 min. The recap from this morning, now in Python: Jay Alammar's house prices
([visual guide](https://jalammar.github.io/visual-interactive-guide-basics-neural-networks/#train_your_dragon)).
One neuron: price = w · area + b. The loss is the mean squared error (MSE): the average of the squared errors.

🧰 **Helpers for Part 0 - just run this cell.** The data (3 houses) and three plotting functions.

In [ ]:
area = np.array([2104.0, 1600.0, 2400.0])  # size of three houses in square feet
price = np.array([399.9, 329.9, 369.0])    # their prices in $1000


def loss_of_w(w, b=0.0):
    """MSE of the line price = w * area + b on the three houses."""
    return np.mean((w * area + b - price) ** 2)


def slope_at(w, tiny=1e-6):
    """Slope of the loss at w, measured by nudging w a tiny bit to the left and to the right."""
    return (loss_of_w(w + tiny) - loss_of_w(w - tiny)) / (2 * tiny)


def plot_line(w=0.1, b=150.0):
    """Plot the houses, the line price = w * area + b, the errors and the MSE."""
    price_hat = w * area + b
    mse = np.mean((price_hat - price) ** 2)
    plt.figure(figsize=(5.5, 3.5))
    plt.scatter(area, price, s=60, zorder=3, label="houses")
    area_line = np.linspace(1400, 2600, 2)
    plt.plot(area_line, w * area_line + b, color="tab:orange", label="model")
    plt.vlines(area, price, price_hat, colors="red", linestyles=":", label="errors")
    plt.ylim(200, 500)
    plt.xlabel("area (sq ft)")
    plt.ylabel("price ($1000)")
    plt.title(f"w = {w:.3f}, b = {b:.0f}  ->  MSE = {mse:,.0f}")
    plt.legend(loc="upper left")
    plt.show()


W_GRID = np.linspace(0.0, 0.35, 200)


def plot_tangent(w_now):
    """The loss as a function of w (b = 0), and the tangent (slope) at w_now."""
    slope = slope_at(w_now)
    plt.figure(figsize=(5.5, 3.5))
    plt.plot(W_GRID, [loss_of_w(w) for w in W_GRID], label="loss L(w)")
    w_tangent = np.linspace(w_now - 0.05, w_now + 0.05, 2)
    plt.plot(w_tangent, loss_of_w(w_now) + slope * (w_tangent - w_now), color="tab:orange", lw=3, label="tangent")
    plt.scatter([w_now], [loss_of_w(w_now)], color="red", zorder=3, s=50)
    plt.xlabel("w")
    plt.ylabel("MSE (b = 0)")
    plt.title(f"slope at w = {w_now}: {slope:,.0f}")
    plt.legend()
    plt.show()


def plot_path(path, lr):
    """The gradient-descent path on the loss curve."""
    plt.figure(figsize=(5.5, 3.5))
    plt.plot(W_GRID, [loss_of_w(w) for w in W_GRID], color="lightgray", label="loss L(w)")
    plt.plot(path, [loss_of_w(w) for w in path], "o-", color="tab:red", label="gradient descent")
    plt.xlabel("w")
    plt.ylabel("MSE (b = 0)")
    plt.title(f"lr = {lr:g}: w = {path[0]} -> {path[-1]:.4f} after {len(path) - 1} steps")
    plt.legend()
    plt.show()

### 0.1 🔧 Train your dragon by hand
This cell shows two sliders for w and b. The red dotted lines are the errors; the title shows the MSE.
Can you get the MSE below 1000? (The best possible line has an MSE of about 447.)

In [ ]:
try:
    from ipywidgets import FloatSlider, interact  # sliders (available in Colab)
    interact(plot_line,
             w=FloatSlider(min=0.0, max=0.3, step=0.005, value=0.1, readout_format=".3f"),
             b=FloatSlider(min=-100.0, max=400.0, step=10.0, value=150.0))
except ImportError:
    print("No sliders available here - change w and b in the line below instead.")
    plot_line(w=0.1, b=150.0)  # 🔧 change w and b

Turning sliders by hand is fine for 2 numbers, but a network has millions. Gradient descent does it for us,
and it only needs one thing: the **slope** of the loss.

### 0.2 ▶️ Derivative = slope
We keep b = 0 and look at the loss as a function of w only. The slope at the current w is measured by
**nudging**: change w a tiny bit and see how much the loss changes. The orange line (the tangent) has this slope.
This cell prints the loss and the slope at `W_NOW` and draws the picture.

In [ ]:
W_NOW = 0.1  # 🔧 try 0.25 and 0.177

print(f"at w = {W_NOW}: loss = {loss_of_w(W_NOW):,.0f}, slope = {slope_at(W_NOW):,.0f}")
print(f"the chain rule from the slides gives the same: mean(2 * error * area) = "
      f"{np.mean(2 * (W_NOW * area - price) * area):,.0f}")
plot_tangent(W_NOW)

🔧 **Change it:** set `W_NOW = 0.25`, then `W_NOW = 0.177`. 🤔 Predict first: is the slope negative, positive or 0?
A negative slope means "the loss goes down if w gets bigger" - so gradient descent makes w bigger.

### 0.3 ✍️ Gradient descent on the loss curve
Gradient descent repeats one line: new w = old w − learning rate × slope. Fill in this line in the loop
(it is the only fill-in of Part 0). The cell then draws the path.
The learning rate is tiny because the areas are large numbers (about 2000) - remember feature scaling from day 1.

In [ ]:
lr = 1e-7    # 🔧 learning rate: try 0.2e-7, 2e-7, 2.5e-7
steps = 10   # 🔧 number of steps
w = 0.1      # start value
path = [w]
for step in range(steps):
    slope = slope_at(w)  # slope of the loss at the current w
    # ✍️ one line: the gradient-descent update of w
    w = w - lr * slope
    path.append(w)
plot_path(path, lr)

This cell checks your line: it runs the same loop once more and compares the result.

In [ ]:
w_reference = 0.1
for step in range(steps):
    w_reference = w_reference - lr * slope_at(w_reference)
check("w after the loop", w, w_reference, "inside the loop: w = w - lr * slope (w does not change yet)", tol=1e-4)

🔧 **Change it:** in the cell above set `lr = 0.2e-7` (slow), `lr = 2e-7` (zig-zag) and `lr = 2.5e-7` (diverges).
🤔 Predict first: which learning rate overshoots the minimum? Which day-1 slide was this?

## Part A — Backprop by hand, checked by PyTorch
⏱ about 25 min. Backprop = the chain rule, one small step at a time, from the loss back to the inputs.
At every node: **gradient = upstream gradient × local slope**. The code does the arithmetic - you read, predict and
change numbers.

🧰 **Helper for Part A - just run this cell.** It draws a chain of nodes with their values and gradients.

In [ ]:
def draw_chain(names, values, grads, title=""):
    """Draw nodes in a row: forward value (green, above) and gradient dL/d(node) (red, below)."""
    fig, ax = plt.subplots(figsize=(2.3 * len(names), 2.2))
    for i in range(len(names)):
        ax.text(i, 0, names[i], ha="center", va="center", fontsize=12,
                bbox={"boxstyle": "round", "facecolor": "#EBEEEE"})
        ax.text(i, 0.55, f"{values[i]:g}", ha="center", color="green", fontsize=13)
        grad_text = "?" if grads[i] is None else f"{grads[i]:g}"
        ax.text(i, -0.75, grad_text, ha="center", color="red", fontsize=13)
        if i < len(names) - 1:
            ax.annotate("", xy=(i + 0.62, 0), xytext=(i + 0.38, 0), arrowprops={"arrowstyle": "->"})
    ax.text(-0.75, 0.55, "value", color="green", ha="right", fontsize=11)
    ax.text(-0.75, -0.75, "dL/d(node)", color="red", ha="right", fontsize=11)
    ax.set_xlim(-1.6, len(names) - 0.5)
    ax.set_ylim(-1.1, 1.0)
    ax.axis("off")
    ax.set_title(title)
    plt.show()

### A.1 ▶️ Worked example: one neuron
The example from the slides: multiply (w · x), add the bias b, ReLU, subtract the target t, square.
Numbers: x = 3, w = 2, b = −1, t = 4. This cell computes the forward pass, then the backward pass line by line,
draws the chain (value green, gradient red) and compares with PyTorch.

In [ ]:
x, w, b, t = 3.0, 2.0, -1.0, 4.0  # 🔧 change the numbers here

# forward: compute and keep every intermediate value
u = w * x          # multiply
v = u + b          # add the bias
r = max(0.0, v)    # ReLU: keeps positive numbers, sets negative ones to 0
e = r - t          # error
L = e ** 2         # loss

# backward: start at the loss and walk back; each line = upstream * local slope
dL_dL = 1.0                               # the loss itself
dL_de = dL_dL * 2 * e                     # L = e^2: local slope 2e
dL_dr = dL_de * 1                         # e = r - t: local slope 1
dL_dv = dL_dr * (1.0 if v > 0 else 0.0)   # ReLU: passes the gradient (1) if v > 0, else blocks it (0)
dL_du = dL_dv * 1                         # v = u + b: a plus node passes the gradient on
dL_db = dL_dv * 1                         # the same for b
dL_dw = dL_du * x                         # u = w * x: local slope with respect to w is x
dL_dx = dL_du * w                         # local slope with respect to x is w

draw_chain(["w", "u = w·x", "v = u + b", "r = ReLU(v)", "e = r - t", "L = e²"],
           [w, u, v, r, e, L], [dL_dw, dL_du, dL_dv, dL_dr, dL_de, dL_dL],
           title=f"x = {x:g}, b = {b:g} (dL/db = {dL_db:g}), t = {t:g}")

# the same with PyTorch: requires_grad=True means "please compute the gradient for me"
x_t = torch.tensor(x, requires_grad=True)
w_t = torch.tensor(w, requires_grad=True)
b_t = torch.tensor(b, requires_grad=True)
L_t = (torch.relu(w_t * x_t + b_t) - t) ** 2  # forward: PyTorch records every operation
L_t.backward()                                # backward: PyTorch applies the chain rule for us
print(f"by hand: dL/dw = {dL_dw:g}, dL/db = {dL_db:g}, dL/dx = {dL_dx:g}")
print(f"PyTorch: dL/dw = {w_t.grad.item():g}, dL/db = {b_t.grad.item():g}, dL/dx = {x_t.grad.item():g}")

🔧 **Change it** in the first line of the cell above (🤔 predict first; set the numbers back afterwards):
1. `b = -7.0`: will dL/dw become 0? Why is this called a **dead ReLU**?
2. `t = 5.0`: now the prediction is perfect. What happens to all gradients?
3. `x = 6.0` and `w = 1.0`: the prediction stays 5, but the input is twice as large. What happens to dL/dw?

### A.2 ✍️ Fill in: a tiny two-layer network
Two neurons in a row, no biases: z1 = w1 · x, h = ReLU(z1), ŷ = w2 · h, L = (ŷ − t)², with x = 1, w1 = 2, w2 = 3,
t = 4. Everything is given except the **last** gradient, dL/dw1: upstream (dL/dz1) × local slope (the slope of
z1 = w1 · x with respect to w1, which is x).

In [ ]:
x, w1, w2, t = 1.0, 2.0, 3.0, 4.0  # ⭐ optional later: try w1 = -2.0

# forward
z1 = w1 * x        # first neuron
h = max(0.0, z1)   # ReLU
y_hat = w2 * h     # second neuron
e = y_hat - t      # error
L = e ** 2         # loss

# backward
dL_dyhat = 2 * e                           # L = e^2 (and e = y_hat - t has local slope 1)
dL_dw2 = dL_dyhat * h                      # y_hat = w2 * h: local slope with respect to w2 is h
dL_dh = dL_dyhat * w2                      # ... and with respect to h it is w2
dL_dz1 = dL_dh * (1.0 if z1 > 0 else 0.0)  # ReLU: 1 if z1 > 0, else 0
dL_dw1 = None  # ✍️ replace: upstream dL_dz1 times the local slope of z1 = w1 * x with respect to w1
dL_dw1 = dL_dz1 * x

draw_chain(["w1", "z1 = w1·x", "h = ReLU(z1)", "ŷ = w2·h", "L = (ŷ - t)²"],
           [w1, z1, h, y_hat, L], [dL_dw1, dL_dz1, dL_dh, dL_dyhat, 1.0],
           title=f"x = {x:g}, w2 = {w2:g} (dL/dw2 = {dL_dw2:g}), t = {t:g}")

This cell checks your line: PyTorch computes the gradient of w1 with autograd.

In [ ]:
w1_t = torch.tensor(w1, requires_grad=True)
L_t = (w2 * torch.relu(w1_t * x) - t) ** 2
L_t.backward()
check("dL_dw1", dL_dw1, w1_t.grad.item(), "dL_dw1 = dL_dz1 * x")

⭐ **optional:** set `w1 = -2.0` in the fill-in cell and run both cells again. 🤔 Predict first: which gradients become 0?

💬 **Explain:** in A.1 the gradient of w was 6 = 2 · 3. Why does the input x appear in the gradient of w?

u = w · x: if we nudge w by a tiny amount, u changes by x times that amount. So the local slope of the multiply
node with respect to w is x, and the gradient is upstream (2) times x (3). A large input makes its weight change a
lot: "error times input", like on day 1. With `x = 6.0, w = 1.0` the prediction and the upstream gradient stay the
same, and dL/dw doubles to 12.

## Part B — Your first MLP in PyTorch: spirals
⏱ about 30 min. Now many neurons at once. PyTorch builds the network for us and does the backward pass.

🧰 **Helpers for Part B - just run this cell.** The spiral data, a function that builds a small network,
a training function (the 4-line training step) and a function that colours the plane by the predicted class.

In [ ]:
def make_spirals(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    """Spiral toy data: X (n, 2) points, y (n,) class labels 0, 1, 2."""
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for k in range(n_classes):
        radius = np.linspace(0.0, 1.0, n_per_class)                    # from the centre outwards
        angle = np.linspace(4 * k, 4 * (k + 1), n_per_class)          # each class turns a bit further
        angle = angle + rng.normal(0, noise, n_per_class)              # some noise
        X_list.append(np.c_[radius * np.sin(angle), radius * np.cos(angle)])
        y_list.append(np.full(n_per_class, k))
    return np.concatenate(X_list), np.concatenate(y_list)


Xs, ys = make_spirals()                              # NumPy arrays (also used in Part E)
X_spiral = torch.tensor(Xs, dtype=torch.float32)     # 300 points, 2 numbers each (x and y position)
y_spiral = torch.tensor(ys)                          # 300 class labels 0, 1, 2


def spiral_model(hidden=100, activation=nn.ReLU):
    """2 inputs -> `hidden` neurons with an activation -> 3 outputs (one score per class)."""
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(2, hidden), activation(), nn.Linear(hidden, 3))


def train_spiral(model, lr=1.0, steps=2000):
    """Train on all 300 points with gradient descent. Returns the loss history."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for step in range(steps):
        optimizer.zero_grad()                              # 1. delete the old gradients
        loss = F.cross_entropy(model(X_spiral), y_spiral)  # 2. forward pass + loss
        loss.backward()                                    # 3. backward pass: all gradients
        optimizer.step()                                   # 4. update the weights
        history.append(loss.item())
    return history


@torch.no_grad()
def spiral_accuracy(model):
    return (model(X_spiral).argmax(dim=1) == y_spiral).float().mean().item()


@torch.no_grad()
def plot_boundary(ax, model, title):
    """Colour the plane by the class the network predicts there, then draw the data on top."""
    grid = np.linspace(-1.2, 1.2, 200)
    grid_x, grid_y = np.meshgrid(grid, grid)
    grid_points = torch.tensor(np.c_[grid_x.ravel(), grid_y.ravel()], dtype=torch.float32)
    grid_class = model(grid_points).argmax(dim=1).numpy().reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, grid_class, levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.3, cmap="viridis")
    ax.scatter(Xs[:, 0], Xs[:, 1], c=ys, s=8, cmap="viridis", edgecolors="k", linewidths=0.2)
    ax.set_title(f"{title}\naccuracy {spiral_accuracy(model):.2f}")
    ax.set_xticks([])
    ax.set_yticks([])

### B.1 ▶️ The data: three spirals
This cell plots the 300 points. No straight line separates the three colours, so a linear model (day 1) fails.

In [ ]:
print("X_spiral:", X_spiral.shape, "| y_spiral:", y_spiral.shape)
plt.figure(figsize=(4, 4))
plt.scatter(Xs[:, 0], Xs[:, 1], c=ys, s=12, cmap="viridis")
plt.title("3 spirals, 300 points")
plt.show()

### B.2 ▶️ Worked example: an MLP and the 4-line training step
This cell builds an MLP with one hidden layer (100 ReLU neurons) and trains it with the same four lines as on the
slides: `zero_grad` → forward + loss → `backward` → `step`. Then it draws the result.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 100),   # input: 2 numbers (x and y position) -> 100 hidden neurons
    nn.ReLU(),           # activation
    nn.Linear(100, 3))   # 100 hidden neurons -> 3 scores, one per class
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)  # gradient descent, learning rate 1.0

for step in range(2000):
    optimizer.zero_grad()                              # 1. delete the old gradients
    loss = F.cross_entropy(model(X_spiral), y_spiral)  # 2. forward pass + loss
    loss.backward()                                    # 3. backward pass: all gradients
    optimizer.step()                                   # 4. update the weights
    if step % 500 == 0:
        print(f"step {step}: loss {loss.item():.3f}")

fig, ax = plt.subplots(figsize=(4, 4))
plot_boundary(ax, model, "100 hidden ReLU neurons")
plt.show()

### B.3 ✍️ Fill in: build the network yourself
Write an `nn.Sequential` with **2 inputs → 10 hidden neurons → ReLU → 3 outputs**
(look at B.2: you only need to change two numbers).

In [ ]:
my_model = None  # ✍️ replace: my_model = nn.Sequential(...)
my_model = nn.Sequential(nn.Linear(2, 10), nn.ReLU(), nn.Linear(10, 3))

This cell checks your network and, if it is correct, trains it and draws the result.

In [ ]:
expected = [("Linear", 2, 10), ("ReLU",), ("Linear", 10, 3)]
found = None
if isinstance(my_model, nn.Sequential):
    found = []
    for layer in my_model:
        if isinstance(layer, nn.Linear):
            found.append(("Linear", layer.in_features, layer.out_features))
        else:
            found.append((type(layer).__name__,))
if found == expected:
    print("✅ correct:", my_model)
    train_spiral(my_model)
    fig, ax = plt.subplots(figsize=(4, 4))
    plot_boundary(ax, my_model, "your network: 10 hidden ReLU neurons")
    plt.show()
else:
    print("❌ not yet - expected nn.Sequential(nn.Linear(2, 10), nn.ReLU(), nn.Linear(10, 3)); found:", found)

### B.4 ▶️ Side by side: no activation, 2, 10 and 100 hidden neurons
This cell trains four networks (a few seconds each) and draws their decisions next to each other.
🤔 **Predict first:** which of the four networks can follow the spirals?

In [ ]:
settings = [(100, nn.Identity, "no activation, H = 100"), (2, nn.ReLU, "ReLU, H = 2"),
            (10, nn.ReLU, "ReLU, H = 10"), (100, nn.ReLU, "ReLU, H = 100")]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for i in range(4):
    hidden, activation, title = settings[i]
    net = spiral_model(hidden, activation)
    train_spiral(net)
    plot_boundary(axes[i], net, title)
plt.tight_layout()
plt.show()

### B.5 🔧 Change it: hidden size, activation, learning rate
Change **one** of the four values at the top of the next cell, run it, and fill in the table.

| change | your prediction | accuracy | what does the boundary look like? |
|---|---|---|---|
| start: `HIDDEN = 100, ACTIVATION = nn.ReLU, LR = 1.0, STEPS = 2000` | | | |
| `HIDDEN = 5` | | | |
| `ACTIVATION = nn.Tanh` | | | |
| `ACTIVATION = nn.Identity` | | | |
| `LR = 0.01` | | | |
| `LR = 10.0` | | | |
| `STEPS = 50` | | | |

This cell trains one network with your values and shows the loss curve and the decision boundary.

In [ ]:
HIDDEN = 100          # 🔧 number of hidden neurons: try 2, 5, 10, 100
ACTIVATION = nn.ReLU  # 🔧 try nn.Tanh, nn.Sigmoid, nn.Identity (= no activation)
LR = 1.0              # 🔧 learning rate: try 0.01, 1.0, 10.0
STEPS = 2000          # 🔧 number of training steps: try 50, 200, 2000

net = spiral_model(HIDDEN, ACTIVATION)
history = train_spiral(net, lr=LR, steps=STEPS)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].plot(history)
axes[0].set(xlabel="step", ylabel="loss (cross-entropy)", title="training loss")
plot_boundary(axes[1], net, f"H = {HIDDEN}, {ACTIVATION.__name__}, lr = {LR}, {STEPS} steps")
plt.tight_layout()
plt.show()

💬 **Explain:** (1) Why does the hidden size matter? (2) Why are all boundaries without an activation straight?

**The table from B.5** (measured; small differences from run to run are normal):

| change | accuracy | what does the boundary look like? |
|---|---|---|
| start: `HIDDEN = 100, ACTIVATION = nn.ReLU, LR = 1.0, STEPS = 2000` | 0.99 | three spirals, made of many short straight pieces |
| `HIDDEN = 5` | 0.65 | only a few straight pieces - too coarse for the spirals |
| `ACTIVATION = nn.Tanh` | 0.99 | smooth, round curves |
| `ACTIVATION = nn.Identity` | 0.54 | straight lines only (the layers collapse into one linear layer) |
| `LR = 0.01` | 0.60 | the right direction, but far from finished |
| `LR = 10.0` | 0.33 | one colour everywhere (= guessing) - the steps overshoot |
| `STEPS = 50` | 0.68 | rough, not finished |

1. Every hidden ReLU neuron adds one straight piece ("kink") to the boundary. With 2 or 5 neurons there are too few
   pieces to follow three spirals (accuracy about 0.50 / 0.65); with 10 or 100 neurons the network fits them
   (about 0.98 / 0.99). `LR = 0.01` (about 0.60) or only 50 steps (about 0.68) - the network has not learned enough
   yet. `LR = 10.0` (about 0.33 = guessing): the steps overshoot and training breaks down (day 1).
2. Without an activation the two layers "collapse" into one linear layer, which can only draw straight lines
   (accuracy about 0.54). Tanh (0.99) and sigmoid (0.96) are non-linear and work.

## Part C — PyTorch MLP on FashionMNIST
⏱ about 35 min. 70 000 grey-scale images (28 × 28 pixels) of 10 clothing classes - a real dataset, the same tools.

### C.1 ▶️ The data
This cell downloads FashionMNIST (about 30 MB) and shows 8 images. We hold out 10 000 training images for
**validation**: we compare our settings on validation data and keep the test set for the very end (day 3).

In [ ]:
from torchvision import datasets

train_raw = datasets.FashionMNIST(root="data", train=True, download=True)
test_raw = datasets.FashionMNIST(root="data", train=False, download=True)
X_all = train_raw.data.float() / 255  # (60000, 28, 28) pixel values between 0 and 1
y_all = train_raw.targets             # (60000,) class numbers 0..9
train_loader = DataLoader(TensorDataset(X_all[:50000], y_all[:50000]), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X_all[50000:], y_all[50000:]), batch_size=1024)
test_loader = DataLoader(TensorDataset(test_raw.data.float() / 255, test_raw.targets), batch_size=1024)
print("training images:", len(train_loader.dataset), "| validation images:", len(val_loader.dataset))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    axes[i].imshow(X_all[i], cmap="gray")
    axes[i].set_title(train_raw.classes[y_all[i]], fontsize=8)
    axes[i].axis("off")
plt.show()

🧰 **Helpers for Parts C and D - just run this cell.**
`make_mlp` builds an MLP with any width and depth, `train_epoch` runs the 4-line training step over all batches
(exactly like in Part B), `evaluate` computes the accuracy, `run_experiment` trains a fresh network and reports it.

In [ ]:
def make_mlp(n_in=784, width=256, depth=2, activation=nn.ReLU, n_out=10):
    """An MLP with `depth` hidden layers of `width` neurons each."""
    layers = [nn.Flatten()]                          # (B, 28, 28) -> (B, 784); does nothing for flat inputs
    size_in = n_in
    for i in range(depth):                           # one hidden layer per loop step
        layers.append(nn.Linear(size_in, width))     # weighted sums
        layers.append(activation())                  # non-linearity
        size_in = width
    layers.append(nn.Linear(size_in, n_out))         # output layer: one score (logit) per class, no activation
    return nn.Sequential(*layers)


def train_epoch(model, loader, optimizer, class_weight=None):
    """One pass over all training batches. Returns the mean training loss."""
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)                          # move the batch to the GPU (if any)
        optimizer.zero_grad()                                          # 1. delete the old gradients
        loss = F.cross_entropy(model(xb), yb, weight=class_weight)     # 2. forward pass + loss
        loss.backward()                                                # 3. backward pass: all gradients
        optimizer.step()                                               # 4. update the weights
        total_loss += loss.item() * len(xb)
        n += len(xb)
    return total_loss / n


@torch.no_grad()  # no gradients needed for evaluation: faster, less memory
def evaluate(model, loader):
    """Fraction of correctly classified images."""
    model.eval()
    correct, n = 0, 0
    for xb, yb in loader:
        predicted_class = model(xb.to(device)).argmax(dim=1)  # the class with the highest score
        correct += (predicted_class == yb.to(device)).sum().item()
        n += len(yb)
    return correct / n


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def run_experiment(width=256, depth=2, activation=nn.ReLU, lr=0.1, epochs=3):
    """Train a new MLP (same seed every time) and return its validation accuracy."""
    torch.manual_seed(0)
    model = make_mlp(width=width, depth=depth, activation=activation).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for epoch in range(epochs):
        train_epoch(model, train_loader, optimizer)
    accuracy = evaluate(model, val_loader)
    print(f"width={width}, depth={depth}, activation={activation.__name__}, lr={lr}: "
          f"{count_parameters(model):,} parameters, validation accuracy {accuracy:.3f}")
    return accuracy


def plot_activations():
    """The four activations (left) and their slopes (right)."""
    z = torch.linspace(-4, 4, 200, requires_grad=True)
    activations = {"ReLU": nn.ReLU(), "Sigmoid": nn.Sigmoid(), "Tanh": nn.Tanh(), "Identity": nn.Identity()}
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for name in activations:
        out = activations[name](z)
        slope = torch.autograd.grad(out.sum(), z)[0]  # the slope of the activation at every z (autograd!)
        axes[0].plot(z.detach(), out.detach(), label=name)
        axes[1].plot(z.detach(), slope, label=name)
    axes[0].set(title="activation(z)", xlabel="z", ylim=(-1.5, 4))
    axes[1].set(title="slope of the activation", xlabel="z")
    axes[0].legend()
    plt.tight_layout()
    plt.show()


def plot_bars(results, title, colors=None):
    """Bar chart of validation accuracies."""
    plt.figure(figsize=(max(5, len(results)), 3))
    plt.bar(list(results.keys()), list(results.values()), color=colors)
    plt.ylim(0, 1)
    plt.ylabel("validation accuracy")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.show()

### C.2 ▶️ Worked example: an MLP on images
This cell builds an MLP 784 → 256 → 256 → 10 (each image is flattened to 784 numbers) and trains it for 3 epochs
with plain gradient descent. Each epoch takes a few seconds on a GPU.

In [ ]:
torch.manual_seed(0)
model = make_mlp().to(device)
print(model)
print("parameters:", count_parameters(model))  # 784*256+256 + 256*256+256 + 256*10+10 = 269 322
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # plain gradient descent, like day 1
for epoch in range(3):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer)
    print(f"epoch {epoch + 1}: training loss {train_loss:.3f}, validation accuracy "
          f"{evaluate(model, val_loader):.3f} ({time.time() - start:.1f} s)")

### C.3 ▶️ Activation functions: what they look like, and a race
The first cell draws the four activations (left) and their slopes (right) - backprop multiplies by these slopes
in every layer.

In [ ]:
plot_activations()

This cell trains the same network with each activation (3 epochs each, about 1 min on a GPU) and compares them.
🤔 **Predict first:** which activation will be worst? Is `nn.Identity` (no activation) much worse than ReLU?

In [ ]:
race = {}
for activation in (nn.ReLU, nn.Sigmoid, nn.Tanh, nn.Identity):
    race[activation.__name__] = run_experiment(activation=activation)
plot_bars(race, "3 epochs of SGD, lr = 0.1", colors=["tab:blue", "tab:orange", "tab:green", "tab:gray"])

### C.4 🔧 Change it: width, depth, learning rate
This cell trains one network. Change **one** argument at a time in it (🤔 predict first) and fill in the table:

| change | your prediction (better / same / worse) | validation accuracy |
|---|---|---|
| start: `width=256, depth=2, lr=0.1` | | |
| `width=32` | | |
| `width=1024` | | |
| `depth=1` | | |
| `depth=4` | | |
| `lr=0.01` | | |
| `lr=1.0` | | |

In [ ]:
run_experiment(width=256, depth=2, activation=nn.ReLU, lr=0.1)  # 🔧 change one argument here

⭐ **optional:** this cell runs the whole table at once and plots it (about 1 min on a GPU).

In [ ]:
all_settings = {"start": {}, "width=32": {"width": 32}, "width=1024": {"width": 1024},
                "depth=1": {"depth": 1}, "depth=4": {"depth": 4}, "lr=0.01": {"lr": 0.01}, "lr=1.0": {"lr": 1.0}}
table = {}
for label in all_settings:
    table[label] = run_experiment(**all_settings[label])
plot_bars(table, "one change at a time")

⭐ **optional: track your experiments with Weights & Biases (W&B).** Instead of writing numbers into the table
by hand, W&B saves the settings and the curves of every run on [wandb.ai](https://wandb.ai), where you can compare
all runs in one plot. You need a free account (students: free academic plan). The first run asks for your API key:
copy it from https://wandb.ai/authorize and paste it into the box. Then open the link printed below the cell.

In [ ]:
!pip install -q wandb
import wandb

wandb.login()  # asks once for your API key (free account, key at https://wandb.ai/authorize)


def run_experiment_wandb(width=256, depth=2, activation=nn.ReLU, lr=0.1, epochs=3):
    """Like run_experiment, but sends the settings and the numbers of every epoch to wandb.ai."""
    config = {"width": width, "depth": depth, "activation": activation.__name__, "lr": lr, "epochs": epochs}
    name = f"width={width}, depth={depth}, {activation.__name__}, lr={lr}"
    with wandb.init(project="srh-dl-day2", config=config, name=name) as run:
        torch.manual_seed(0)
        model = make_mlp(width=width, depth=depth, activation=activation).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
        for epoch in range(epochs):
            train_loss = train_epoch(model, train_loader, optimizer)
            accuracy = evaluate(model, val_loader)
            run.log({"epoch": epoch + 1, "train_loss": train_loss, "val_accuracy": accuracy})
        run.summary["parameters"] = count_parameters(model)
    return accuracy


for label in all_settings:
    run_experiment_wandb(**all_settings[label], epochs=10)  # 🔧 or one run: run_experiment_wandb(width=32)

💬 **Explain:** (1) Which settings are clearly worst, and why? (2) Do width and depth matter much here?
(3) Would you trust a difference of 0.3 percentage points?

**The table from C.4** (validation accuracy after 3 epochs, measured):

| change | validation accuracy |
|---|---|
| start: `width=256, depth=2, lr=0.1` | 0.81 |
| `width=32` | 0.81 (same) |
| `width=1024` | 0.82 (same) |
| `depth=1` | 0.82 (same) |
| `depth=4` | 0.77 (worse) |
| `lr=0.01` | 0.67 (worse - too slow) |
| `lr=1.0` | 0.10 (much worse - guessing) |

1. `lr=1.0` (about 0.10 = guessing among 10 classes): the steps are too large and training breaks down.
   `nn.Sigmoid` (about 0.58): its slope is at most 0.25 (right plot), so with plain SGD it learns slowly (see C.5).
   `lr=0.01` (about 0.67) is simply too slow for 3 epochs.
   Surprise: `nn.Identity` (no activation, about 0.81) is almost as good as ReLU here. The layers do collapse into
   one linear layer, but a linear model already does well on FashionMNIST. With longer training the ReLU network
   pulls ahead, and on the spirals (Part B) the linear network fails completely.
2. Less than you might expect: widths 32, 256 and 1024 and depth 1 all end up at about 0.81-0.82. Depth 4 is a bit
   worse after 3 epochs (about 0.77): deeper networks need more training or better tricks (day 3).
3. No. One seed, one split, 3 epochs: such differences are within the run-to-run noise. Repeat with several seeds
   (day 3).

### C.5 ▶️ Demo: vanishing gradients in a deep sigmoid MLP
🧰 **Just run the next two cells.** Two networks with 10 layers each, the same start weights and the same batch;
only the activation differs. Left: the size of the gradient in every layer after one backward pass.
Right: the accuracy after one epoch of training.

In [ ]:
def deep_mlp(activation):
    """9 hidden layers + 1 output layer = 10 linear layers, with He start weights (day 3)."""
    torch.manual_seed(0)
    model = make_mlp(width=128, depth=9, activation=activation).to(device)
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    return model


def vanishing_gradient_demo():
    xb, yb = next(iter(train_loader))  # one batch of 256 images
    xb, yb = xb.to(device), yb.to(device)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
    accuracy_after_1_epoch = {}
    for activation in (nn.Sigmoid, nn.ReLU):
        model = deep_mlp(activation)
        F.cross_entropy(model(xb), yb).backward()  # one backward pass at the start
        norms = [layer.weight.grad.norm().item() for layer in model if isinstance(layer, nn.Linear)]
        axes[0].semilogy(range(1, 11), norms, "o-", label=activation.__name__)
        print(f"{activation.__name__:>7}: gradient size layer 1 = {norms[0]:.1e}, layer 10 = {norms[-1]:.1e} "
              f"(layer 10 / layer 1 = {norms[-1] / norms[0]:,.1f})")
        model = deep_mlp(activation)  # fresh network, then one epoch of training
        train_epoch(model, train_loader, torch.optim.SGD(model.parameters(), lr=0.05))
        accuracy_after_1_epoch[activation.__name__] = evaluate(model, val_loader)
    axes[0].set(xlabel="layer (1 = next to the input)", ylabel="gradient size (log scale)",
                title="gradient size per layer at the start")
    axes[0].legend()
    axes[1].bar(list(accuracy_after_1_epoch.keys()), list(accuracy_after_1_epoch.values()),
                color=["tab:blue", "tab:orange"])
    axes[1].set(ylim=(0, 1), ylabel="validation accuracy", title="after 1 epoch of SGD")
    plt.tight_layout()
    plt.show()

This cell runs the demo (about 30 s on a GPU, a few minutes on a CPU).

In [ ]:
vanishing_gradient_demo()

💬 **Explain:** with sigmoid, the gradient in layer 1 is thousands of times smaller than in layer 10. Why does the
deep sigmoid network hardly learn in one epoch?

Backprop multiplies one slope per layer, and the slope of the sigmoid is at most 0.25. After 9 layers almost nothing
is left (here about 13 000 times smaller), so the first layers hardly change and the network stays close to guessing
(about 10 % for 10 classes). ReLU has slope 0 or 1 and does not shrink the gradient. Fixes: day 3 and day 4.

## Part D — MLP on EEG band-power features (CHB-MIT)
⏱ about 25 min (what you do not finish is homework).

Patient chb01, 6 recordings with one seizure each, cut into 4-s windows. For every window and each of the
18 channels we compute how much power there is in 5 frequency bands (delta, theta, alpha, beta, low gamma)
→ 18 × 5 = **90 features** per window. An MLP on such features is a classical, strong baseline.

**Record-wise split:** records 0-3 train, record 4 validation, record 5 test. Neighbouring 4-s windows look almost
the same, so a random split of windows would leak information from training to test (day 3).

### D.1 ▶️ The data (download about 250 MB, 1-2 min)
This cell installs the EEG library `mne` (in Colab; about 30 s).

In [ ]:
!pip install -q mne

🧰 This cell contains the course's EEG helper functions (`load_chbmit`, `band_power`) - just run it.

In [ ]:
"""Small loaders for the two open-access EEG seizure datasets used in the labs.

CHB-MIT  (Shoeb 2009, PhysioNet): paediatric scalp EEG, 23 bipolar channels, 256 Hz, 1 h records.
SeizeIT2 (Bhagubai et al. 2025, OpenNeuro ds005873): wearable behind-the-ear EEG, 2 channels, 256 Hz.
Files are downloaded once and cached in ./data (in Colab: lost when the runtime is recycled).
"""
import re
import urllib.request
from pathlib import Path

import numpy as np

DATA = Path("data")
CHBMIT = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"  # AWS mirror of PhysioNet, much faster
SEIZEIT2 = "https://s3.amazonaws.com/openneuro.org/ds005873/"
# the first 18 CHB-MIT channels form the longitudinal bipolar "double banana" montage
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4",
            "F4-C4", "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]


def fetch(url, name):
    """Download url to data/name once; return the local path."""
    out = DATA / name
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", url)
        urllib.request.urlretrieve(url, out.with_suffix(".part"))
        out.with_suffix(".part").rename(out)  # no half-written files in the cache
    return out


def preprocess(raw, picks, fs):
    """Band-pass 0.5-40 Hz, resample to fs, return (channels, time) in microvolts."""
    raw.pick(picks).filter(0.5, 40, verbose="error").resample(fs, verbose="error")
    return raw.get_data() * 1e6


def windows(data, fs, win_s, events, around_s=None):
    """Cut (channels, time) into non-overlapping windows of win_s seconds.

    y = 1 if the window centre lies inside a seizure event (onset_s, end_s).
    around_s: keep only windows within around_s seconds of a seizure (None = keep all).
    """
    n = int(win_s * fs)
    k = data.shape[1] // n
    X = data[:, :k * n].reshape(data.shape[0], k, n).transpose(1, 0, 2)
    mid = (np.arange(k) + 0.5) * win_s
    y = np.zeros(k, dtype=np.int64)
    keep = np.zeros(k, dtype=bool) if around_s is not None else np.ones(k, dtype=bool)
    for a, b in events:
        y[(mid >= a) & (mid < b)] = 1
        if around_s is not None:
            keep |= (mid >= a - around_s) & (mid < b + around_s)
    return np.ascontiguousarray(X[keep], dtype=np.float32), y[keep]


def chbmit_seizures(patient="chb01"):
    """{'chb01_03': [(2996, 3036)], ...} parsed from the PhysioNet summary file."""
    text = fetch(f"{CHBMIT}{patient}/{patient}-summary.txt", f"chbmit/{patient}-summary.txt").read_text()
    out = {}
    for block in text.split("File Name: ")[1:]:
        t = [int(s) for s in re.findall(r"Seizure.*(?:Start|End) Time:\s*(\d+)", block)]
        out[block.split(".edf")[0]] = list(zip(t[::2], t[1::2]))
    return out


def load_chbmit(records=("chb01_03", "chb01_04", "chb01_15", "chb01_16", "chb01_18", "chb01_26"),
                win_s=4.0, fs=128):
    """CHB-MIT records -> X (n, 18, win_s*fs) in uV, y (n,) 1 = seizure, rec (n,) index into records."""
    import mne
    X, y, rec = [], [], []
    for i, r in enumerate(records):
        patient = r.split("_")[0]
        raw = mne.io.read_raw_edf(fetch(f"{CHBMIT}{patient}/{r}.edf", f"chbmit/{r}.edf"), preload=True, verbose="error")
        raw.rename_channels(lambda c: c.removesuffix("-0"))  # MNE renames the duplicated T8-P8 channel
        Xi, yi = windows(preprocess(raw, CHANNELS, fs), fs, win_s, chbmit_seizures(patient)[r])
        X.append(Xi), y.append(yi), rec.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(rec)


def seizeit2_events(subject="sub-001"):
    """{run_path: [(onset_s, end_s), ...]} seizure events of every EEG run of one SeizeIT2 subject."""
    listing = urllib.request.urlopen(
        f"https://s3.amazonaws.com/openneuro.org?list-type=2&prefix=ds005873/{subject}/").read().decode()
    out = {}
    for stem in re.findall(r"<Key>ds005873/([^<]+?/eeg/[^<]+?)_events\.tsv</Key>", listing):
        rows = fetch(f"{SEIZEIT2}{stem}_events.tsv", f"seizeit2/{stem}_events.tsv").read_text().splitlines()[1:]
        cols = [r.split("\t") for r in rows]
        out[stem] = [(float(c[0]), float(c[0]) + float(c[1])) for c in cols if c[2].startswith("sz")]
    return out


def load_seizeit2(subject="sub-001", win_s=4.0, fs=128, around_s=1800):
    """Seizure-containing runs of one SeizeIT2 subject -> X (n, 2, win_s*fs) in uV, y (n,), run (n,).

    around_s keeps +-30 min around each seizure, otherwise one subject is >100 h of mostly background.
    """
    import mne
    X, y, run = [], [], []
    for i, (stem, events) in enumerate(e for e in seizeit2_events(subject).items() if e[1]):
        raw = mne.io.read_raw_edf(fetch(f"{SEIZEIT2}{stem}_eeg.edf", f"seizeit2/{stem}_eeg.edf"), preload=True, verbose="error")
        Xi, yi = windows(preprocess(raw, raw.ch_names[:2], fs), fs, win_s, events, around_s)
        X.append(Xi), y.append(yi), run.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(run)


def band_power(X, fs=128):
    """Log band power (delta, theta, alpha, beta, low gamma) per channel: (n, ch, t) -> (n, ch*5)."""
    from scipy.signal import welch
    f, p = welch(X, fs=fs, nperseg=fs, axis=-1)
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 40)]
    bp = np.stack([p[..., (f >= a) & (f < b)].mean(-1) for a, b in bands], -1)
    return np.log(bp + 1e-10).reshape(len(X), -1).astype(np.float32)

This cell downloads 6 recordings of patient chb01, cuts them into 4-s windows and computes the 90 band-power
features per window.

In [ ]:
X, y, rec = load_chbmit()  # X (5081, 18, 512) in µV, y (5081,) 1 = seizure, rec (5081,) recording 0..5
X_bp = band_power(X)       # (5081, 90) = 18 channels x 5 bands, log power
print("raw windows", X.shape, "-> features", X_bp.shape, f"| seizure windows: {y.sum()} of {len(y)}")
for r in range(6):
    print(f"recording {r}: {np.sum(rec == r)} windows, {y[rec == r].sum()} seizure windows")

This cell splits by recording and standardises the features with the mean and standard deviation of the
**training** recordings only (day 1).

In [ ]:
train, val, test = rec <= 3, rec == 4, rec == 5               # whole recordings, never mixed
mean, std = X_bp[train].mean(axis=0), X_bp[train].std(axis=0)  # statistics from the training recordings only
X_std = ((X_bp - mean) / std).astype(np.float32)             # standardise all windows with them
X_train, y_train = torch.tensor(X_std[train]), torch.tensor(y[train])
X_val, y_val = torch.tensor(X_std[val]), torch.tensor(y[val])
X_test, y_test = X_std[test], y[test]                        # test data stays NumPy for the metrics
print("windows: train", len(y_train), "| validation", len(y_val), "| test", len(y_test))

🧰 **Helpers for Part D - just run this cell.** `train_eeg_model` trains an MLP 90 → 64 → 64 → 2 for 30 epochs
(with `make_mlp` and `train_epoch` from Part C) and predicts the test recording; `show_results` prints the
accuracy and draws the confusion matrix.

In [ ]:
def train_eeg_model(class_weight=None, epochs=30):
    """MLP 90 -> 64 -> 64 -> 2, trained with SGD. Returns the model and its predictions on the test recording."""
    torch.manual_seed(0)
    model = make_mlp(n_in=90, width=64, depth=2, n_out=2).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    if class_weight is not None:
        class_weight = torch.tensor(class_weight, dtype=torch.float32, device=device)
    for epoch in range(epochs):
        train_loss = train_epoch(model, loader, optimizer, class_weight)
        if (epoch + 1) % 10 == 0:
            val_acc = evaluate(model, DataLoader(TensorDataset(X_val, y_val), batch_size=1024))
            print(f"epoch {epoch + 1}: training loss {train_loss:.4f}, validation accuracy {val_acc:.3f}")
    model.eval()
    with torch.no_grad():
        scores = model(torch.tensor(X_test).to(device))  # (n_test, 2) scores
    return model, scores.argmax(dim=1).cpu().numpy()  # predicted class per test window


def show_results(pred, title):
    """Accuracy and confusion matrix on the test recording."""
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    print(f"{title}: test accuracy {np.mean(pred == y_test):.3f} | "
          f"found {cm[1, 1]} of {cm[1].sum()} seizure windows | false alarms: {cm[0, 1]} of {cm[0].sum()} background windows")
    ConfusionMatrixDisplay(cm, display_labels=["background", "seizure"]).plot(cmap="Blues")
    plt.title(title)
    plt.show()

### D.2 ▶️ Worked example: an MLP seizure detector
This cell trains the detector (about 30 s) and shows its results on the test recording.

In [ ]:
eeg_model, pred = train_eeg_model()
show_results(pred, "MLP on band power")

How to read the confusion matrix: rows = the truth, columns = the prediction. Bottom right = seizure windows the
model found, bottom left = seizure windows it missed, top right = false alarms.

### D.3 ✍️ Fill in: a "detector" that never raises an alarm
🤔 **Predict first:** what accuracy does a detector reach that **always** says "no seizure"?
Create one prediction (0 = no seizure) per test window.

In [ ]:
pred_never = None  # ✍️ replace: an array of zeros, one per test window (np.zeros(..., dtype=int))
pred_never = np.zeros(len(y_test), dtype=int)

This cell checks your line and, if it is correct, shows the results of the "never alarm" detector.

In [ ]:
check("number of predictions", None if pred_never is None else len(pred_never), len(y_test),
      "one prediction per test window: np.zeros(len(y_test), dtype=int)")
if pred_never is not None and len(pred_never) == len(y_test):
    show_results(pred_never, "never says 'seizure'")

### D.4 ⭐ optional — Change it: class weights
Only a few windows are seizures, so the loss hardly cares about them. A **class weight** makes every seizure window
count more in the loss. This cell trains the detector again with a weight of 20 for seizures.
🔧 Change `20.0` to `5.0` or `50.0`. 🤔 Predict first: what happens to the found seizure windows and the false alarms?

In [ ]:
weighted_model, pred_weighted = train_eeg_model(class_weight=[1.0, 20.0])  # 🔧 change the seizure weight
show_results(pred_weighted, "MLP with class weights")

💬 **Explain:** only 86 of the 5081 windows (1.7 %) are seizures. Why is accuracy a misleading number here?
Which numbers would you report instead?

A detector that never raises an alarm is already right for every background window: about 98.3 % accuracy on all
data and about 95.5 % on the test recording - without finding a single seizure. So a high accuracy says almost
nothing about the seizures. Better: how many seizure windows or seizures were found (sensitivity), false alarms per
hour, and always the confusion matrix (day 3). Class weights usually trade more found seizures for more false
alarms; here the MLP already finds 25 of 26 seizure windows, so weight 20 mainly adds false alarms (10 -> 17).

## Summary
- A derivative is a slope; the chain rule multiplies the small slopes along the way: **upstream × local**.
- Backprop = the chain rule from the loss back to every weight. PyTorch's autograd does exactly this.
- An MLP = Linear layers + an activation. Without the activation (`nn.Identity`) it collapses to a linear model.
- Every PyTorch training step: `zero_grad` → forward + loss → `backward` → `step`.
- Deep sigmoid networks suffer from vanishing gradients; ReLU helps.
- On imbalanced EEG data, accuracy is misleading: look at the confusion matrix.

**Outlook:** almost every model ends with such an MLP "head": a few Linear layers trained with cross-entropy and
backprop. The CNNs of day 4 - and your challenge model - end exactly like this.

## ⭐ Optional Part E — Backprop by hand in NumPy (for the curious)
Only if you want to see what PyTorch does inside. We write the forward and backward pass of the spiral MLP from
Part B ourselves in NumPy and check that our gradients are the same as PyTorch's.

🧰 **Helpers - just run this cell.** Start weights, ReLU, softmax and cross-entropy in NumPy.

In [ ]:
def init_params(n_in=2, n_hidden=100, n_out=3, seed=0):
    """Random start weights (shape (out, in), like PyTorch) and zero biases."""
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_in), (n_hidden, n_in)),  # hidden layer weights (H, 2)
        "b1": np.zeros(n_hidden),                                   # hidden layer biases (H,)
        "W2": rng.normal(0, 0.01, (n_out, n_hidden)),               # output layer weights (3, H)
        "b2": np.zeros(n_out),                                      # output layer biases (3,)
    }


def relu(Z):
    return np.maximum(0, Z)  # element-wise: negative numbers become 0


def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)    # subtract the row maximum: same result, no overflow in exp
    expZ = np.exp(Z)
    return expZ / expZ.sum(axis=1, keepdims=True)  # every row now sums to 1 = probabilities


def cross_entropy(P, Y):
    """Mean cross-entropy: -log(probability of the correct class), averaged over the batch."""
    return -np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))


def numerical_grad(loss_function, array, tiny=1e-6):
    """Nudge every entry of `array` up and down and measure how the loss changes."""
    grad = np.zeros_like(array)
    for index in np.ndindex(array.shape):
        old = array[index]
        array[index] = old + tiny
        loss_plus = loss_function()
        array[index] = old - tiny
        loss_minus = loss_function()
        array[index] = old  # put the old value back!
        grad[index] = (loss_plus - loss_minus) / (2 * tiny)
    return grad

### E.1 ▶️ Forward and backward pass: the "six lines"
This cell defines the forward pass (store the intermediate values) and the backward pass: the "six lines" from the
backup slides. Every gradient has the **same shape** as its parameter.

In [ ]:
def forward(X, params):
    """X (B, 2) -> probabilities P (B, 3) and a cache of the intermediate values for the backward pass."""
    Z1 = X @ params["W1"].T + params["b1"]   # hidden layer, before the activation (B, H)
    A1 = relu(Z1)                            # hidden layer, after ReLU (B, H)
    Z2 = A1 @ params["W2"].T + params["b2"]  # output layer: one score (logit) per class (B, 3)
    P = softmax(Z2)                          # probabilities (B, 3)
    return P, {"X": X, "Z1": Z1, "A1": A1, "P": P}


def backward(cache, Y, params):
    """Gradients of the mean cross-entropy for all parameters."""
    B = len(Y)                                       # batch size
    D2 = (cache["P"] - Y) / B                        # output error: prediction minus target (B, 3)
    dW2 = D2.T @ cache["A1"]                         # "error times input" of the output layer (3, H)
    db2 = D2.sum(axis=0)                             # bias: sum of the errors over the batch (3,)
    D1 = (D2 @ params["W2"]) * (cache["Z1"] > 0)     # back through W2, then the ReLU "gate" (B, H)
    dW1 = D1.T @ cache["X"]                          # "error times input" of the hidden layer (H, 2)
    db1 = D1.sum(axis=0)                             # (H,)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


Ys = np.eye(3)[ys]  # one-hot targets, e.g. class 2 -> [0, 0, 1]
params = init_params()
P, cache = forward(Xs, params)
grads = backward(cache, Ys, params)
print(f"loss at the start = {cross_entropy(P, Ys):.4f}   (guessing 1/3 for every class gives ln 3 = {np.log(3):.4f})")
for name in params:
    print(f"{name}: parameter shape {params[name].shape}, gradient shape {grads[name].shape}")

### E.2 ▶️ Trust, but check
This cell compares our gradients with (1) nudging every single weight and (2) PyTorch autograd, on a tiny network
(10 points, 8 hidden neurons) because nudging costs two forward passes **per weight**.

In [ ]:
X_tiny, y_tiny = Xs[::30], ys[::30]  # every 30th point -> 10 points
Y_tiny = np.eye(3)[y_tiny]
rng = np.random.default_rng(1)
p_tiny = init_params(n_hidden=8, seed=1)
for name in p_tiny:
    p_tiny[name] = p_tiny[name] + 0.5 * rng.normal(size=p_tiny[name].shape)  # random values everywhere


def tiny_loss():
    return cross_entropy(forward(X_tiny, p_tiny)[0], Y_tiny)


grads_backprop = backward(forward(X_tiny, p_tiny)[1], Y_tiny, p_tiny)

p_torch = {}
for name in p_tiny:
    p_torch[name] = torch.tensor(p_tiny[name], requires_grad=True)  # float64 like NumPy
hidden = torch.relu(torch.tensor(X_tiny) @ p_torch["W1"].T + p_torch["b1"])
logits = hidden @ p_torch["W2"].T + p_torch["b2"]
F.cross_entropy(logits, torch.tensor(y_tiny)).backward()

for name in p_tiny:
    diff_nudging = np.abs(grads_backprop[name] - numerical_grad(tiny_loss, p_tiny[name])).max()
    diff_torch = np.abs(grads_backprop[name] - p_torch[name].grad.numpy()).max()
    print(f"{name}: max difference to nudging {diff_nudging:.1e}, to PyTorch {diff_torch:.1e}")
    assert diff_nudging < 1e-6 and diff_torch < 1e-10

Differences of about $10^{-9}$ or smaller mean: our six lines compute exactly what PyTorch computes.

### E.3 ✍️ Fill in: the ReLU slope as a 0/1 mask
In `backward` the part `(cache["Z1"] > 0)` is the slope of ReLU: 1 where the neuron was active (> 0), else 0.
Compute it for a small example.

In [ ]:
Z_small = np.array([[1.5, -0.3, 0.0],
                    [-2.0, 0.7, 4.0]])
relu_slope = None  # ✍️ replace: an array of 1.0 where Z_small > 0, else 0.0 (same shape as Z_small)
relu_slope = (Z_small > 0).astype(float)

This cell checks your line.

In [ ]:
check("relu_slope", relu_slope, [[1, 0, 0], [0, 1, 1]],
      "(Z_small > 0) gives True/False; .astype(float) turns it into 1.0/0.0. Note: at exactly 0 the slope is 0")